In [ ]:
library(dplyr)
library(ggplot2)
library(tidyr)
library(readr)
library(tibble)
library(purrr)
library(stringr)
library(forcats)

In [ ]:
#library(tidyverse)
library(janitor)
library(vegan)       # for RDA

In [ ]:
# read saved metadata
meta_final <- read.csv("251206_MetaFinal_Complete.csv")

In [ ]:
glimpse(meta_final)

In [ ]:
expr_raw <- read.csv("20250710_RNana_Seqtk_Filtered_vst_limma_w.DesignMat.csv")
head(expr_raw)

In [ ]:
library(dplyr)

In [ ]:
clean_names <- function(x) {
  x %>%
    str_replace("_[Rr]edo.*$", "") %>%    # remove _redo, _redo1, _Redo2 etc.
    str_replace("_[Vv][0-9]+$", "") %>%   # remove _v1, _V2...
    str_replace("_[Rr][0-9]+$", "") %>%   # remove _r1, _R2...
    str_replace("_[0-9]+$", "")           # remove pure numeric suffixes like _1
}

In [ ]:
expr_clean <- expr_raw
colnames(expr_clean) <- colnames(expr_clean) %>% clean_names()

In [ ]:
colnames(expr_clean)[1:20]

In [ ]:
expr_ids <- colnames(expr_clean)[-1]

In [ ]:
length(expr_ids)   # should be 616
head(expr_ids)

In [ ]:
meta_clean <- meta_final %>%
  mutate(sample_clean = clean_names(sample))

In [ ]:
head(meta_clean$sample_clean)

In [ ]:
meta_616 <- meta_clean %>%
  filter(sample_clean %in% expr_ids) %>%
  distinct(sample_clean, .keep_all = TRUE)

In [ ]:
nrow(meta_616)
length(unique(meta_616$sample_clean))

In [ ]:
meta_final_616<- tibble(sample_clean = expr_ids) %>%
  left_join(meta_616, by = "sample_clean")

In [ ]:
dim(meta_final_616)                     # should be 616 × (metadata columns)
identical(meta_final_616$sample_clean, expr_ids)   # TRUE

In [ ]:
glimpse(meta_final_616)

In [ ]:
expr_clean <- expr_raw
colnames(expr_clean) <- colnames(expr_clean) %>% clean_names()

In [ ]:
colnames(expr_clean)[1:20]

In [ ]:
expr_samples <- colnames(expr_clean)[-1]   # skip gene column
meta_samples <- meta_final_616$sample          # clean, final, correct sample IDs

In [ ]:
intersect(expr_samples, meta_samples) %>% length()
setdiff(meta_samples, expr_samples)        # meta but not expression
setdiff(expr_samples, meta_samples)        # expression but not meta

In [ ]:
common_samples <- intersect(expr_samples, meta_samples)

expr_mat <- expr_clean %>%
  select(gene_id = 1, all_of(common_samples))


In [ ]:
expr_mat <- expr_clean %>%
  select(gene_id = 1, all_of(meta_final_616$sample))

In [ ]:
gene_ids <- expr_mat$gene_id

expr_num <- expr_mat %>%
  select(-gene_id) %>%
  as.matrix()

storage.mode(expr_num) <- "double"

dim(expr_num)   # genes × 616 samples


In [ ]:
# PCA

In [ ]:
# PCA needs samples as rows → transpose
expr_log_t <- t(expr_num) # Transpose because PCA expects samples as rows

# should be 616 × (number of genes)


In [ ]:
sum(is.na(expr_log_t))

In [ ]:
pca_res <- prcomp(expr_log_t, center = TRUE, scale. = TRUE) #run PCA

In [ ]:
pca_var  <- pca_res$sdev^2
pca_prop <- pca_var / sum(pca_var)

# print variance of the first few PCs
round(100 * pca_prop[1:10], 2)


In [ ]:
library(dplyr)

pca_df <- as_tibble(pca_res$x) %>%
  mutate(sample = rownames(pca_res$x)) %>%
  left_join(meta_final, by = "sample")


In [ ]:
library(ggplot2)
library(viridis)

p_pca12 <- ggplot(pca_df,
                  aes(PC1, PC2,
                      colour = country,
                      shape  = season_short)) +
  geom_point(size = 2.2, alpha = 0.85) +
  scale_color_manual(
  values = c(
    "DE" = "black",
    "US" = "steelblue",
    "FR" = "indianred2"
  ),
  name = "Country"
)+
coord_cartesian(
    xlim = c(-300, 300),
    ylim = c(-100, 100)
  ) +
  theme_bw() +
  theme(
    panel.grid      = element_blank(),
    legend.position = "top"
  ) +
  labs(
    title  = "",
    x      = sprintf("PC1 (%.1f%%)", 100 * pca_prop[1]),
    y      = sprintf("PC2 (%.1f%%)", 100 * pca_prop[2]),
    colour = "Country",
    shape  = "Season"
  )

p_pca12


In [ ]:
## Save as PNG (for drafts / slides)
ggsave(
  filename = "Fig3A_PCA_GeneExpression616.pdf",
  plot     = p_pca12,
  width    = 184,
  height   = 184,
  units    = "mm",
  dpi      = 300
)

In [ ]:
# RDA

In [ ]:
install.packages("vegan")

In [ ]:
library(vegan)

In [ ]:
expr_rda_t <- t(expr_num)
dim(expr_rda_t)   # should be 616 × ~18000 genes

In [ ]:
# Abiotic (weather)
abiotic_vars <- c("mean_temp_4d", "mean_hum_4d",
                  "mean_fluctuation_4d", "precip_30d_sum", "hours_after_sunrise")

# Plant morphology
morpho_vars <- c("max_diam_mm", "n_leaves")

# Presence absence

p_a_vars <- c("bolting","hr")

# Composite field scores
composite_vars <- c("composite_bacterial", "composite_fungi_hpa")

# Microbial load (metagenomic)
microbe_vars <- c("bacterial_per_plant", "fungal_per_plant", "bact_family_entropy")

# Field symptoms
symptom_vars <- c("herbivory")

vars_all <- c(abiotic_vars, morpho_vars,
              composite_vars, microbe_vars,
              symptom_vars, p_a_vars)

In [ ]:
numeric_cols <- sapply(vars_all, is.numeric)

In [ ]:
head(numeric_cols)

In [ ]:
env <- meta_final_616 %>% select(all_of(vars_all))

In [ ]:
# Coerce everything in env to numeric (they came in as <chr> from CSV)
env <- env %>%
  mutate(across(everything(), as.numeric))

In [ ]:
# Quick sanity check
str(env)
colSums(is.na(env))
summary(env)

In [ ]:
## 4) (Optional) Remove rows with any NA in env ----
# If you still have a few NAs and want to drop those samples:
keep <- complete.cases(env)
sum(!keep)      # how many samples would be dropped?

In [ ]:
env_complete   <- env[keep, , drop = FALSE]
expr_rda_use   <- expr_rda_t[keep, , drop = FALSE]

In [ ]:
## 5) Scale environmental variables ----
env_scaled <- as.data.frame(scale(env_complete, center = TRUE, scale = TRUE))

In [ ]:
library(vegan)

## 6) Run RDA ----
rda_res <- rda(expr_rda_use ~ ., data = env_scaled)
rda_res

In [ ]:
RsquareAdj(rda_res)

In [ ]:
# sample (site) scores
site_scores <- as_tibble(scores(rda_res, display = "sites", scaling = 2)) %>%
  mutate(sample = meta_final$sample)

# environmental (arrow) scores
bp_scores <- as_tibble(scores(rda_res, display = "bp", scaling = 2),
                       rownames = "variable")


In [ ]:
glimpse(bp_scores)

In [ ]:
library(ggrepel)

In [ ]:
library(vegan)
library(ggplot2)
library(ggrepel)
library(grid)

# -------------------------
# Extract variance explained
# -------------------------
sm <- summary(rda_res)
vx <- sm$cont$importance[2, 1:2] * 100

# -------------------------
# Scores
# -------------------------
sites <- as.data.frame(scores(rda_res, display = "sites", scaling = 2))
bipl  <- as.data.frame(scores(rda_res, display = "bp", scaling = 2))
bipl$Trait <- rownames(bipl)

# -------------------------
# Rescale arrows
# -------------------------
asc <- max(abs(sites$RDA1), abs(sites$RDA2)) /
       max(abs(bipl$RDA1),  abs(bipl$RDA2))

bipl$RDA1s <- bipl$RDA1 * asc
bipl$RDA2s <- bipl$RDA2 * asc

# -------------------------
# Rename traits
# -------------------------
trait_labels <- c(
  mean_temp_4d        = "Temp (4d mean)",
  mean_hum_4d         = "Humidity (4d mean)",
  mean_fluctuation_4d = "Temp fluctuation (4d)",
  precip_30d_sum      = "Precipitation (30d)",
  hours_after_sunrise = "Hours after sunrise",
  max_diam_mm         = "Max leaf diameter",
  n_leaves            = "Leaf number",
  composite_bacterial = "Bacterial composite",
  composite_fungi_hpa = "Fungal composite",
  bacterial_per_plant = "Bacterial reads",
  fungal_per_plant    = "Fungal reads",
  bact_family_entropy = "Bacterial diversity",
  herbivory           = "Herbivory",
  bolting             = "Bolting",
  hr                  = "HR"
)

bipl$Trait_clean <- trait_labels[bipl$Trait]
bipl$Trait_clean[is.na(bipl$Trait_clean)] <- bipl$Trait[is.na(bipl$Trait_clean)]

# -------------------------
# Optional: keep strongest arrows only (top 70%)
# -------------------------
bipl$len <- sqrt(bipl$RDA1s^2 + bipl$RDA2s^2)
#bipl_use <- subset(bipl, len > quantile(len, 0.0))

# -------------------------
# Journal theme
# -------------------------
theme_journal <- function() {
  theme_bw(base_size = 8, base_family = "Arial") +
    theme(
      panel.grid = element_blank(),
      panel.grid.major.x = element_blank(),
      axis.title = element_text(size = 8),
      axis.text  = element_text(size = 6)
    )
}

# -------------------------
# RDA plot
# -------------------------
p_rda <- ggplot() +
  geom_point(data = sites,
             aes(RDA1, RDA2),
             color = "grey70",
             size = 1.6,
             alpha = 0.7) +

  geom_segment(data = bipl_use,
               aes(x = 0, y = 0,
                   xend = RDA1s, yend = RDA2s),
               arrow = arrow(length = unit(2, "mm")),
               linewidth = 0.3) +

coord_cartesian(
    xlim = c(-30, 30),
    ylim = c(-25, 25)
  ) +

  ggrepel::geom_text_repel(data = bipl_use,
               aes(x = RDA1s * 1.15,
                   y = RDA2s * 1.15,
                   label = Trait_clean),
               size = 2.6,
               family = "Arial",
               max.overlaps = Inf) +

  labs(
    x = paste0("RDA1 (", round(vx[1], 1), "% of constrained)"),
    y = paste0("RDA2 (", round(vx[2], 1), "% of constrained)")
  ) +

  theme_journal()

# -------------------------
# View
# -------------------------
p_rda


In [ ]:
ggsave("Fig2D_Inset_RDA.pdf",
       p_rda,
       width = 92,
       height = 92,
       units = "mm",
       device = cairo_pdf)

In [ ]:
# enforce metadata order to match expression sample order
meta_final_616 <- meta_final_616 %>%
  slice(match(colnames(expr_num), sample))

stopifnot(identical(meta_final_616$sample, colnames(expr_num)))

expr_rda_t <- t(expr_num)
rownames(expr_rda_t) <- meta_final_616$sample

stopifnot(identical(rownames(expr_rda_t), meta_final_616$sample))


In [ ]:
library(ggplot2)
library(ggrepel)
library(vegan)

run_rda_subset <- function(idx, label) {
  # subset expression + env
  expr_sub <- expr_rda_t[idx, , drop = FALSE]
  env_sub  <- env_scaled[idx, , drop = FALSE]
  
  # run RDA
  rda_sub <- rda(expr_sub ~ ., data = env_sub)
  
  # summary + constrained variance
  sm <- summary(rda_sub)
  vx <- sm$cont$importance[2, 1:2] * 100
  
  # scores
  sites <- as.data.frame(scores(rda_sub, display = "sites", scaling = 2))
  bipl  <- as.data.frame(scores(rda_sub, display = "bp",    scaling = 2))
  bipl$Trait <- rownames(bipl)
  
  # arrow scaling
  asc <- max(abs(sites$RDA1), abs(sites$RDA2)) /
         max(abs(bipl$RDA1),  abs(bipl$RDA2))
  
  # biplot
  p <- ggplot() +
    geom_point(data = sites,
               aes(RDA1, RDA2),
               color = "grey70",
               size  = 1.6,
               alpha = 0.7) +
    geom_segment(data = bipl,
                 aes(x = 0, y = 0,
                     xend = RDA1 * asc,
                     yend = RDA2 * asc),
                 arrow = arrow(length = unit(0.25, "cm"))) +
    ggrepel::geom_text_repel(data = bipl,
                             aes(x = RDA1 * asc * 1.2,
                                 y = RDA2 * asc * 1.2,
                                 label = Trait),
                             size = 3.2) +
    labs(
      title = paste0("RDA – ", label),
      x = paste0("RDA1 (", round(vx[1], 1), "%)"),
      y = paste0("RDA2 (", round(vx[2], 1), "%)")
    ) +
    theme_minimal(base_size = 12)
  
  list(rda = rda_sub, plot = p)
}


In [ ]:
idx_autumn <- grepl("Autumn", meta_final_616$season)
idx_spring <- grepl("Spring", meta_final_616$season)

table(idx_autumn, idx_spring)


In [ ]:
res_autumn <- run_rda_subset(idx_autumn, "Autumn")
res_spring <- run_rda_subset(idx_spring, "Spring")

p_rda_autumn <- res_autumn$plot
p_rda_spring <- res_spring$plot

p_rda_autumn
p_rda_spring


In [ ]:
library(patchwork)

p_rda_seasons <- p_rda_autumn | p_rda_spring +
  plot_annotation(title = "RDA by season (all genes)")

p_rda_seasons


In [ ]:
## Save as PNG (for drafts / slides)
ggsave(
  filename = "Fig3B_RDA_Seasons.png",
  plot     = p_rda_seasons,
  width    = 10,
  height   = 5,
  dpi      = 300
)